# 01 — Pre-training safety scoring

Two safety dimensions, **Content Safety** and **Physical Safety**, each with
sub-dimensions that carry a real, stated per-record evaluation criterion.

What this notebook does, per dataset, **one dataset at a time**:

1. reads the dataset's **modality** and **context** from `dataset_specs.py`;
2. keeps only the sub-dimensions that are (a) toggled on, (b) valid for that
   modality, (c) valid for that context, and (d) have every required input —
   everything else is **N/A with a stated reason**, never scored as zero;
3. scores **every record** against each surviving sub-dimension's criterion;
4. rolls each sub-dimension up to **its own** dataset score and compares it
   to **its own** threshold, chosen by the risk level you declare.

### Two rules this notebook will not break

* **No composite.** There is no S(D). Sub-dimensions are never averaged with
  each other, and datasets are never pooled. Every score is
  (one dataset x one sub-dimension).
* **Fractions only.** Doses, scores and thresholds are all in `[0, 1]`.
  `safety_lib.unit_check` raises if anything arrives as a percent.

In [1]:
import os
import sys

sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd

import safety_lib as sl
from dataset_specs import ALL_SPECS, BY_NAME, TABULAR_SPECS, TEXT_SPECS

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 60)
print(f"safety_lib {sl.VERSION} | units = {sl.UNITS}")

safety_lib 2.0-notebooks | units = fraction


## The sub-dimension catalog

`record_criterion` is what is evaluated on each individual record.
`dataset_aggregation` is how those record verdicts become the one number for
the dataset. `dose_validated` marks the sub-dimensions that notebook 02 puts
through a controlled dose-response check.

In [2]:
cat = sl.catalog_frame()
print(cat.to_string(index=False))
sl.write_csv(cat, "01_subdimension_catalog.csv", n_preview=len(cat))

                      subdimension       dimension                                    label modalities                                contexts          requires                                                        record_criterion                                     dataset_aggregation                                                   detector  dose_validated
              harm_content_density  content_safety                  Harmful content density       text                                     all       text_column                                     record toxicity probability >= 0.50    fraction of records at or above the record threshold                                   Detoxify (toxicity head)            True
           identity_attack_density  content_safety                  Identity-attack density       text                                     all       text_column                              record identity-attack probability >= 0.50    fraction of records at or above the 

WindowsPath('D:/Documents/PhD_program/DISSERTATION_RELATED/Dissertation_Studies_code/safety_notebooks/notebooks/results/01_subdimension_catalog.csv')

---
# INPUT 1 — toggles

Flip any sub-dimension to `False` to exclude it from this run. A sub-dimension
turned off here shows up in the applicability trace as
*"toggled off by operator"*, so an excluded sub-dimension is always visible
in the output rather than silently missing.

In [3]:
TOGGLES = {
    # ---- Content Safety ----
    "harm_content_density":               True,
    "identity_attack_density":            True,
    "severe_toxicity_density":            True,
    "label_integrity":                    True,
    "free_text_field_harm":               True,
    "representation_imbalance":           True,
    # ---- Physical Safety ----
    "physical_harm_enablement":           True,
    "threat_density":                     True,
    "measurement_range_violation":        True,
    "safety_critical_edge_case_coverage": True,
    "outcome_severity_exposure":          True,
}
assert set(TOGGLES) == set(sl.BY_ID), "TOGGLES must cover exactly the catalog"

# INPUT 2 — context and risk level, per dataset

**Context** decides which sub-dimensions are even meaningful. It is declared
on the dataset in `dataset_specs.py`; override it here if you want to score
the same file under a different context.

Available contexts: `health`, `loan_finance`, `transportation`, `biology`,
`chemistry`, `academic`, `general`.

**Risk level** picks the tolerance column from the threshold table below.
`high` = the dataset feeds a high-consequence decision, so the tolerance is
tightest.

In [4]:
RUN = {
    #  dataset name    : (context override or None, risk level)
    "diabetes_130":  (None,   "high"),      # clinical readmission -> high stakes
    "framingham":    (None,   "high"),      # cardiovascular risk  -> high stakes
    "german_credit": (None,   "medium"),    # lending decision
    "civilcomments": (None,   "medium"),    # general-purpose text corpus
}

# Set to the datasets you actually want to score in this run.
DATASETS_TO_SCORE = ["diabetes_130", "framingham", "german_credit", "civilcomments"]

# INPUT 3 — thresholds by risk level

One tolerance per sub-dimension per risk level, as a fraction. A dataset
**fails** a sub-dimension when its score for that sub-dimension exceeds the
tolerance. Edit any number below; `THRESHOLD_OVERRIDES` lets you pin a single
sub-dimension without touching the table.

In [5]:
THRESHOLDS = {
    # subdimension                        high     medium    low
    "harm_content_density":              {"high": 0.010, "medium": 0.050, "low": 0.100},
    "identity_attack_density":           {"high": 0.005, "medium": 0.020, "low": 0.050},
    "severe_toxicity_density":           {"high": 0.001, "medium": 0.005, "low": 0.020},
    "label_integrity":                   {"high": 0.050, "medium": 0.100, "low": 0.200},
    "free_text_field_harm":              {"high": 0.010, "medium": 0.050, "low": 0.100},
    "representation_imbalance":          {"high": 0.100, "medium": 0.200, "low": 0.350},
    "physical_harm_enablement":          {"high": 0.005, "medium": 0.020, "low": 0.050},
    "threat_density":                    {"high": 0.005, "medium": 0.020, "low": 0.050},
    "measurement_range_violation":       {"high": 0.010, "medium": 0.050, "low": 0.100},
    "safety_critical_edge_case_coverage":{"high": 0.100, "medium": 0.250, "low": 0.400},
    "outcome_severity_exposure":         {"high": 0.100, "medium": 0.250, "low": 0.500},
}

# e.g. {"label_integrity": 0.03} to pin one sub-dimension for every dataset
THRESHOLD_OVERRIDES: dict[str, float] = {}

tf = sl.threshold_frame(THRESHOLDS)
print(tf.to_string(index=False))
sl.write_csv(tf, "01_thresholds.csv", n_preview=len(tf))

                      subdimension  high  medium  low
              harm_content_density 0.010   0.050 0.10
           identity_attack_density 0.005   0.020 0.05
           severe_toxicity_density 0.001   0.005 0.02
                   label_integrity 0.050   0.100 0.20
              free_text_field_harm 0.010   0.050 0.10
          representation_imbalance 0.100   0.200 0.35
          physical_harm_enablement 0.005   0.020 0.05
                    threat_density 0.005   0.020 0.05
       measurement_range_violation 0.010   0.050 0.10
safety_critical_edge_case_coverage 0.100   0.250 0.40
         outcome_severity_exposure 0.100   0.250 0.50

[csv] D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\results\01_thresholds.csv   (11 rows x 4 cols, units=fraction)
                      subdimension  high  medium  low
              harm_content_density 0.010   0.050 0.10
           identity_attack_density 0.005   0.020 0.05
           severe_tox

WindowsPath('D:/Documents/PhD_program/DISSERTATION_RELATED/Dissertation_Studies_code/safety_notebooks/notebooks/results/01_thresholds.csv')

## Applicability trace

Before scoring anything: for each dataset, which sub-dimensions run and —
for the ones that do not — exactly why. This is the answer to *"what happened
to the other sub-dimensions?"*

In [6]:
appl = pd.concat([sl.applicability(BY_NAME[n], TOGGLES) for n in DATASETS_TO_SCORE],
                 ignore_index=True)
grid = appl.pivot_table(index="subdimension", columns="dataset",
                        values="applicable", aggfunc="first")
print("\napplicable? (True = scored for that dataset)\n")
print(grid.to_string())
sl.write_csv(appl, "01_applicability_matrix.csv",
             cols=["dataset", "context", "dimension", "subdimension",
                   "enabled", "applicable", "na_reason"], n_preview=20)


applicable? (True = scored for that dataset)

dataset                             civilcomments  diabetes_130  framingham  german_credit
subdimension                                                                              
free_text_field_harm                        False         False       False          False
harm_content_density                         True         False       False          False
identity_attack_density                      True         False       False          False
label_integrity                             False          True        True           True
measurement_range_violation                 False          True        True          False
outcome_severity_exposure                   False          True        True          False
physical_harm_enablement                     True         False       False          False
representation_imbalance                    False          True        True           True
safety_critical_edge_case_coverage         

WindowsPath('D:/Documents/PhD_program/DISSERTATION_RELATED/Dissertation_Studies_code/safety_notebooks/notebooks/results/01_applicability_matrix.csv')

## Text detector

The text sub-dimensions need Detoxify. There is **no fallback classifier** —
if it is missing the run fails loudly rather than quietly scoring with
something else. Skip this cell if you are scoring tabular datasets only.

In [7]:
TEXT_SCORER = None
if any(BY_NAME[n].modality == "text" or BY_NAME[n].text_column
       for n in DATASETS_TO_SCORE):
    TEXT_SCORER = sl.DetoxifyScorer("unbiased")
    print(f"text detector: {TEXT_SCORER.name} on {TEXT_SCORER.device}")

D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.3-alpha/toxic_debiased-c7548aa0.ckpt" to C:\Users\joyce/.cache\torch\hub\checkpoints\toxic_debiased-c7548aa0.ckpt


100%|██████████| 476M/476M [01:03<00:00, 7.84MB/s] 
D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\joyce\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading we

text detector: detoxify:unbiased on cpu


## Score each dataset — alone

One loop iteration = one dataset. Nothing crosses between iterations.

This takes some time to run 35 min + because of the CivilComments dataset.

In [8]:
subdim_all = []

for name in DATASETS_TO_SCORE:
    spec = BY_NAME[name]
    ctx_override, risk = RUN[name]
    if ctx_override:
        spec = sl.replace(spec, context=ctx_override)

    print("\n" + "=" * 78)
    print(f"DATASET {spec.name}   modality={spec.modality}   context={spec.context}"
          f"   risk_level={risk}")
    print("=" * 78)

    df = spec.load()
    print(f"loaded {df.shape[0]} records x {df.shape[1]} columns")

    subs, recs = sl.score_dataset(
        df, spec,
        risk_level=risk,
        toggles=TOGGLES,
        thresholds=THRESHOLDS,
        threshold_overrides=THRESHOLD_OVERRIDES,
        text_scorer=TEXT_SCORER,
        seed=0,
        keep_record_scores=True,
    )

    # per-record scores for this dataset (long: one row per record x sub-dimension)
    sl.write_csv(recs, f"01_record_scores__{sl.slug(spec.name)}.csv",
                 cols=["dataset", "record_id", "dimension", "subdimension",
                       "record_risk", "record_flag"])

    # the sub-dimension scorecard for this dataset
    sl.write_csv(subs, f"01_subdimension_scores__{sl.slug(spec.name)}.csv",
                 cols=["dataset", "context", "dimension", "subdimension",
                       "applicable", "n_records", "n_flagged", "score",
                       "risk_level", "threshold", "exceeds_threshold"],
                 n_preview=len(subs))
    subdim_all.append(subs)


DATASET diabetes_130   modality=tabular   context=health   risk_level=high
loaded 101766 records x 46 columns

[csv] D:\Documents\PhD_program\DISSERTATION_RELATED\Dissertation_Studies_code\safety_notebooks\notebooks\results\01_record_scores__diabetes_130.csv   (508830 rows x 7 cols, units=fraction)
     dataset  record_id      dimension    subdimension  record_risk  record_flag
diabetes_130          0 content_safety label_integrity     0.016619        False
diabetes_130          1 content_safety label_integrity     0.075231        False
diabetes_130          2 content_safety label_integrity     0.091310        False
diabetes_130          3 content_safety label_integrity     0.121432        False
diabetes_130          4 content_safety label_integrity     0.075088        False
diabetes_130          5 content_safety label_integrity     0.069375        False
diabetes_130          6 content_safety label_integrity     0.063095        False
diabetes_130          7 content_safety label_integr

## Scorecards

One row per (dataset, sub-dimension). Read each row on its own — these
numbers are deliberately not combined.

In [9]:
allsubs = pd.concat(subdim_all, ignore_index=True)
view = allsubs.loc[allsubs["applicable"] & allsubs["enabled"],
                   ["dataset", "context", "dimension", "subdimension", "n_records",
                    "n_flagged", "score", "risk_level", "threshold",
                    "exceeds_threshold"]]
print(view.to_string(index=False))

sl.write_csv(allsubs, "01_subdimension_scores__ALL.csv",
             cols=["dataset", "dimension", "subdimension", "applicable",
                   "score", "threshold", "exceeds_threshold"],
             n_preview=len(allsubs))

      dataset      context       dimension                       subdimension  n_records  n_flagged    score risk_level  threshold exceeds_threshold
 diabetes_130       health  content_safety                    label_integrity   101766.0    11371.0 0.191372       high      0.050              True
 diabetes_130       health  content_safety           representation_imbalance   101766.0      644.0 0.650836       high      0.100              True
 diabetes_130       health physical_safety        measurement_range_violation   101766.0        0.0 0.000000       high      0.010             False
 diabetes_130       health physical_safety safety_critical_edge_case_coverage   101766.0       87.0 0.165200       high      0.100              True
 diabetes_130       health physical_safety          outcome_severity_exposure   101766.0    11357.0 0.111599       high      0.100              True
   framingham       health  content_safety                    label_integrity     4240.0      622.0 0.2319

WindowsPath('D:/Documents/PhD_program/DISSERTATION_RELATED/Dissertation_Studies_code/safety_notebooks/notebooks/results/01_subdimension_scores__ALL.csv')

### Sub-dimensions over their threshold

The actionable output of this notebook: which dataset fails which
sub-dimension, at the risk level declared for it.

In [10]:
fails = allsubs[allsubs["exceeds_threshold"] == True]
if len(fails):
    print(fails[["dataset", "context", "risk_level", "dimension", "subdimension",
                 "score", "threshold", "n_flagged", "n_records"]].to_string(index=False))
else:
    print("no sub-dimension exceeded its threshold at the declared risk levels")
sl.write_csv(
    allsubs.assign(
        verdict=np.where(allsubs["exceeds_threshold"] == True, "EXCEEDS",
                 np.where(allsubs["exceeds_threshold"] == False, "within",
                          "N/A"))
    )[["dataset", "context", "risk_level", "dimension", "subdimension",
       "score", "threshold", "verdict", "na_reason", "units"]],
    "01_verdicts.csv", n_preview=len(allsubs))

      dataset      context risk_level       dimension                       subdimension    score  threshold  n_flagged  n_records
 diabetes_130       health       high  content_safety                    label_integrity 0.191372       0.05    11371.0   101766.0
 diabetes_130       health       high  content_safety           representation_imbalance 0.650836       0.10      644.0   101766.0
 diabetes_130       health       high physical_safety safety_critical_edge_case_coverage 0.165200       0.10       87.0   101766.0
 diabetes_130       health       high physical_safety          outcome_severity_exposure 0.111599       0.10    11357.0   101766.0
   framingham       health       high  content_safety                    label_integrity 0.231990       0.05      622.0     4240.0
   framingham       health       high  content_safety           representation_imbalance 0.244575       0.10      110.0     4240.0
   framingham       health       high physical_safety        measurement_range_viol

WindowsPath('D:/Documents/PhD_program/DISSERTATION_RELATED/Dissertation_Studies_code/safety_notebooks/notebooks/results/01_verdicts.csv')

### Guard: no composite was produced

A deliberate assertion, so a future edit cannot quietly reintroduce an
averaged score across sub-dimensions or across datasets.

In [11]:
assert allsubs.groupby(["dataset", "subdimension"]).size().max() == 1, \
    "each (dataset, sub-dimension) must appear exactly once"
assert "composite" not in " ".join(allsubs.columns).lower()
print("OK — one score per (dataset, sub-dimension); no composite, no pooling.")
print(f"units = {sl.UNITS} everywhere; scores, thresholds and doses are fractions in [0,1].")

OK — one score per (dataset, sub-dimension); no composite, no pooling.
units = fraction everywhere; scores, thresholds and doses are fractions in [0,1].
